In [1]:

import sys, subprocess, hashlib, json
from pathlib import Path
import importlib.metadata as md

import torch
import torchvision
import numpy as np

assert torch.cuda.is_available(), "Activer le GPU dans Kaggle"

REPO = Path("/kaggle/working/resilientinspect-ai")
COMMIT = "d27569d9d0e25659252b33a204007377920ceec6"

if not (REPO / ".git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/nouhabenkhelil/resilientinspect-ai.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", "main"],
        check=True
    )

subprocess.run(
    ["git", "-C", str(REPO), "checkout", COMMIT],
    check=True
)

# Protéger les bibliothèques GPU préinstallées
protected = {
    name: md.version(name)
    for name in ("torch", "torchvision", "numpy")
}

constraints = Path("/kaggle/working/transfer_constraints.txt")
constraints.write_text(
    "".join(f"{k}=={v}\n" for k, v in protected.items())
)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-q", "-e", f"{REPO}[train,eval,dev]",
    "-c", str(constraints)
], check=True)

assert protected == {
    name: md.version(name) for name in protected
}

sys.path.insert(0, str(REPO / "src"))

print("GPU :", torch.cuda.get_device_name(0))
print("PyTorch :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("Commit :", COMMIT[:10])


Cloning into '/kaggle/working/resilientinspect-ai'...
Updating files: 100% (13739/13739), done.
Note: switching to 'd27569d9d0e25659252b33a204007377920ceec6'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at d27569d Add B0 A1 statistical diagnostic
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installe

GPU : Tesla T4
PyTorch : 2.11.0+cu130
Torchvision : 0.26.0+cu130
Commit : d27569d9d0


In [2]:
import torch
import torchvision
import mlflow
import sklearn
import yaml
import matplotlib

from torchvision.models import efficientnet_b0

model = efficientnet_b0(weights=None)

print("PyTorch :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))
print("EfficientNet-B0 : chargement réussi")
print("MLflow :", mlflow.__version__)
print("Scikit-learn :", sklearn.__version__)
print("Matplotlib :", matplotlib.__version__)

del model

PyTorch : 2.11.0+cu130
Torchvision : 0.26.0+cu130
CUDA disponible : True
GPU : Tesla T4
EfficientNet-B0 : chargement réussi
MLflow : 3.16.1
Scikit-learn : 1.8.0
Matplotlib : 3.10.0


In [3]:

matches = list(Path("/kaggle/input").rglob("bottle/train/good"))
assert len(matches) == 1, "Dataset MVTec absent ou ambigu"

DATA_ROOT = matches[0].parents[2]
MANIFEST = REPO / "data/manifests/manifest.csv"

if not MANIFEST.exists():
    subprocess.run([
        sys.executable, "-m",
        "resilientinspect.data.prepare",
        "--root", str(DATA_ROOT),
        "--output-dir", str(MANIFEST.parent),
        "--report", "/kaggle/working/transfer_preparation.json"
    ], cwd=REPO, check=True)

EXPECTED_SHA = (
    "13f55c7cf7c87a3e3739e24920bb0d85cc96"
    "a39a00e30d277ddfaa1832b84522"
)

assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == EXPECTED_SHA

print("Dataset :", DATA_ROOT)
print("Manifestes identiques à B0/A1 : OK")


{
  "counts": {
    "bottle": {
      "train": 167,
      "val": 42,
      "test": 83,
      "test_good": 20,
      "test_anomalous": 63
    },
    "hazelnut": {
      "train": 312,
      "val": 79,
      "test": 110,
      "test_good": 40,
      "test_anomalous": 70
    },
    "leather": {
      "train": 196,
      "val": 49,
      "test": 124,
      "test_good": 32,
      "test_anomalous": 92
    }
  },
  "totals": {
    "train": 675,
    "val": 170,
    "test": 317
  },
  "manifest_sha256": {
    "manifest.csv": "13f55c7cf7c87a3e3739e24920bb0d85cc96a39a00e30d277ddfaa1832b84522",
    "train.csv": "781b1143fee9eab14f871f4f6335a94f9832c8bb0f13efbb8c40f54d3eadc5d4",
    "val.csv": "08f10f23fa04a531188684a46ae2578c9337e8c81f6c3c20aaf1b3c20b963092",
    "test.csv": "10b4d50e66d9c98cfd53a55b304be4fe150ad4e1eaf7e7a99c1c019d793d30ac"
  }
}
Dataset : /kaggle/input/datasets/nouhebenkhelil/resilientinspect-mvtec3
Manifestes identiques à B0/A1 : OK


In [4]:
import sys
import site
import importlib
from pathlib import Path

REPO = Path("/kaggle/working/resilientinspect-ai")

# Vérifier que le fichier existe
assert (REPO / "src/models/transfer.py").is_file(), \
    "Le fichier transfer.py est absent"

# Recharger les chemins des packages installés
for folder in site.getsitepackages():
    site.addsitedir(folder)

importlib.invalidate_caches()

# Tester l'importation
from resilientinspect.models.transfer import (
    _load_imagenet_state,
    IMAGENET_URL
)

print("Importation Transfer Learning réussie !")
print("URL ImageNet :", IMAGENET_URL)

Importation Transfer Learning réussie !
URL ImageNet : https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth


In [5]:

from resilientinspect.models.transfer import (
    _load_imagenet_state,
    IMAGENET_URL
)

filename = IMAGENET_URL.rsplit("/", 1)[-1]
mounted = list(Path("/kaggle/input").rglob(filename))

assert len(mounted) <= 1, "Plusieurs fichiers ImageNet trouvés"

state, provenance = _load_imagenet_state(
    mounted[0] if mounted else None,
    allow_download=not bool(mounted)
)

IMAGENET = Path(provenance["local_path"])
del state

print("Poids ImageNet :", IMAGENET)
print("SHA256 :", provenance["sha256"])
print("Vérification officielle réussie")


Poids ImageNet : /kaggle/working/resilientinspect-ai/artifacts/transfer_weights/efficientnet_b0_rwightman-7f5810bc.pth
SHA256 : 7f5810bc96def8f7552d5b7e68d53c4786f81167d28291b21c0d90e1fca14934
Vérification officielle réussie


In [6]:

PLAN = Path("/kaggle/working/transfer_data/seed42_plan.json")
PLAN.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m",
    "resilientinspect.data.synthetic",
    "--root", str(DATA_ROOT),
    "--manifest", str(MANIFEST),
    "--config", str(REPO / "configs/synthetic_transfer.yaml"),
    "--seed", "42",
    "--output", str(PLAN)
], cwd=REPO, check=True)

plan = json.loads(PLAN.read_text())

assert len(plan["recipes"]) == 1690

print("Plan synthétique généré : OK")
print("Recettes :", len(plan["recipes"]))
print("Catégories :", plan["categories"])


{"output": "/kaggle/working/transfer_data/seed42_plan.json", "sha256": "b1bd73bc639561e335b9997c33369d2611c2fff5d0df996f77c1fba0883ce8f4", "recipes": 1690, "categories": ["bottle", "hazelnut", "leather"], "inference_executed": false, "images_opened": false}
Plan synthétique généré : OK
Recettes : 1690
Catégories : ['bottle', 'hazelnut', 'leather']


In [7]:

RUNS = Path("/kaggle/working/transfer_runs")

for variant in ["T0", "T1", "T2"]:
    print(f"\n=== Smoke GPU {variant} ===")

    command = [
        sys.executable, "-m",
        "resilientinspect.training.train_transfer",
        "--config", str(REPO / f"configs/train_{variant.lower()}.yaml"),
        "--category", "bottle",
        "--seed", "42",
        "--root", str(DATA_ROOT),
        "--manifest", str(MANIFEST),
        "--synthetic-plan", str(PLAN),
        "--output-dir", str(RUNS),
        "--device", "cuda",
        "--smoke"
    ]

    if variant != "T0":
        command += ["--imagenet-weights", str(IMAGENET)]

    subprocess.run(command, cwd=REPO, check=True)
    print(variant, ": smoke réussi")

print("\nLes trois smoke tests GPU sont terminés.")



=== Smoke GPU T0 ===


2026/11/13 22:46:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/13 22:46:11 INFO mlflow.store.db.utils: Updating database tables


{"variant": "T0", "category": "bottle", "scope": "smoke", "epoch": 1, "train_loss": 0.7452977895736694, "train_accuracy": 0.75, "train_auroc": 0.0, "train_ap": 0.25, "train_f1": 0.0, "train_images": 4, "train_batches": 2, "train_tn": 3, "train_tp": 0, "train_fp": 0, "train_fn": 1, "val_loss": 0.6939550638198853, "val_accuracy": 0.5, "val_auroc": 0.5, "val_ap": 0.5, "val_f1": 0.0, "val_images": 4, "val_batches": 2, "val_tn": 2, "val_tp": 0, "val_fp": 0, "val_fn": 2}
{"run_id": "da6758e1aad8479881f6eb92f37087e1", "output": "/kaggle/working/transfer_runs/T0/bottle/seed42/smoke_da6758e1aad8479881f6eb92f37087e1", "variant": "T0", "scope": "smoke"}
T0 : smoke réussi

=== Smoke GPU T1 ===
{"variant": "T1", "category": "bottle", "scope": "smoke", "epoch": 1, "train_loss": 0.80414879322052, "train_accuracy": 0.25, "train_auroc": 0.33333333333333337, "train_ap": 0.3333333333333333, "train_f1": 0.4, "train_images": 4, "train_batches": 2, "train_tn": 0, "train_tp": 1, "train_fp": 3, "train_fn": 0,

In [8]:

from zipfile import ZipFile, ZIP_DEFLATED

ARCHIVE = Path("/kaggle/working/transfer_full_backup.zip")

def backup_models():
    temporary = ARCHIVE.with_suffix(".zip.tmp")

    with ZipFile(temporary, "w", ZIP_DEFLATED) as z:
        z.write(
            PLAN,
            arcname="transfer_data/seed42_plan.json"
        )

        for run in RUNS.glob("T*/*/seed42/full_*"):
            if not (run / "summary.json").is_file():
                continue

            for file in run.rglob("*"):
                if file.is_file():
                    z.write(
                        file,
                        arcname=file.relative_to(
                            Path("/kaggle/working")
                        ).as_posix()
                    )

    temporary.replace(ARCHIVE)
    print("Archive actualisée :", ARCHIVE)
    print("Taille Mo :", round(ARCHIVE.stat().st_size / 1024**2, 2))


def train_transfer(variant, category):
    assert variant in ("T0", "T1", "T2")
    assert category in ("bottle", "hazelnut", "leather")

    directory = RUNS / variant / category / "seed42"
    completed = []

    for file in directory.glob("full_*/summary.json"):
        summary = json.loads(file.read_text())

        if (summary.get("status") == "completed"
            and summary.get("variant") == variant
            and summary.get("category") == category
            and summary.get("scope") == "full"
            and summary.get("test_used") is False
            and (file.parent / "best.pt").is_file()):
            completed.append(file)

    assert len(completed) <= 1, "Plusieurs runs complets : revue requise"

    if completed:
        print(f"{variant}/{category} : déjà terminé")
        return

    command = [
        sys.executable, "-m",
        "resilientinspect.training.train_transfer",
        "--config", str(REPO / f"configs/train_{variant.lower()}.yaml"),
        "--category", category,
        "--seed", "42",
        "--root", str(DATA_ROOT),
        "--manifest", str(MANIFEST),
        "--synthetic-plan", str(PLAN),
        "--output-dir", str(RUNS),
        "--device", "cuda"
    ]

    if variant != "T0":
        command += ["--imagenet-weights", str(IMAGENET)]

    subprocess.run(command, cwd=REPO, check=True)

    print(f"{variant}/{category} : entraînement complet réussi")
    backup_models()


In [9]:
for category in ["bottle", "hazelnut", "leather"]:
    train_transfer("T0", category)

{"variant": "T0", "category": "bottle", "scope": "full", "epoch": 1, "train_loss": 0.5504342533514172, "train_accuracy": 0.7125748502994012, "train_auroc": 0.7707339811395174, "train_ap": 0.8136410185160363, "train_f1": 0.6643356643356644, "train_images": 334, "train_batches": 21, "train_tn": 143, "train_tp": 95, "train_fp": 24, "train_fn": 72, "val_loss": 0.784451961517334, "val_accuracy": 0.5, "val_auroc": 0.5, "val_ap": 0.5, "val_f1": 0.0, "val_images": 84, "val_batches": 6, "val_tn": 42, "val_tp": 0, "val_fp": 0, "val_fn": 42}
{"variant": "T0", "category": "bottle", "scope": "full", "epoch": 2, "train_loss": 0.3306377471207145, "train_accuracy": 0.8832335329341318, "train_auroc": 0.9128688730323783, "train_ap": 0.9385617816061123, "train_f1": 0.8729641693811075, "train_images": 334, "train_batches": 21, "train_tn": 161, "train_tp": 134, "train_fp": 6, "train_fn": 33, "val_loss": 0.789900643484933, "val_accuracy": 0.5, "val_auroc": 0.3441043083900227, "val_ap": 0.4281709137388958, "

In [10]:
for category in ["bottle", "hazelnut", "leather"]:
    train_transfer("T1", category)

{"variant": "T1", "category": "bottle", "scope": "full", "epoch": 1, "train_loss": 0.6279912044902047, "train_accuracy": 0.7245508982035929, "train_auroc": 0.8123632973573811, "train_ap": 0.769516411298848, "train_f1": 0.7371428571428571, "train_images": 334, "train_batches": 21, "train_tn": 113, "train_tp": 129, "train_fp": 54, "train_fn": 38, "val_loss": 0.5224051702590216, "val_accuracy": 0.8809523809523809, "val_auroc": 0.9988662131519274, "val_ap": 0.998917748917749, "val_f1": 0.8648648648648649, "val_images": 84, "val_batches": 6, "val_tn": 42, "val_tp": 32, "val_fp": 0, "val_fn": 10}
{"variant": "T1", "category": "bottle", "scope": "full", "epoch": 2, "train_loss": 0.4687058035128131, "train_accuracy": 0.9491017964071856, "train_auroc": 0.9932948474308867, "train_ap": 0.9943845320775405, "train_f1": 0.9463722397476341, "train_images": 334, "train_batches": 21, "train_tn": 167, "train_tp": 150, "train_fp": 0, "train_fn": 17, "val_loss": 0.3935234404745556, "val_accuracy": 0.97619

In [11]:
for category in ["bottle", "hazelnut", "leather"]:
    train_transfer("T2", category)

{"variant": "T2", "category": "bottle", "scope": "full", "epoch": 1, "train_loss": 0.39156953464011235, "train_accuracy": 0.874251497005988, "train_auroc": 0.9658288213991179, "train_ap": 0.9686999770780644, "train_f1": 0.8679245283018868, "train_images": 334, "train_batches": 21, "train_tn": 154, "train_tp": 138, "train_fp": 13, "train_fn": 29, "val_loss": 0.23277350550606138, "val_accuracy": 0.9285714285714286, "val_auroc": 1.0, "val_ap": 1.0, "val_f1": 0.9230769230769231, "val_images": 84, "val_batches": 6, "val_tn": 42, "val_tp": 36, "val_fp": 0, "val_fn": 6}
{"variant": "T2", "category": "bottle", "scope": "full", "epoch": 2, "train_loss": 0.046961159428615055, "train_accuracy": 0.9970059880239521, "train_auroc": 0.9998924307074474, "train_ap": 0.9998943289890809, "train_f1": 0.996996996996997, "train_images": 334, "train_batches": 21, "train_tn": 167, "train_tp": 166, "train_fp": 0, "train_fn": 1, "val_loss": 0.00493096886202693, "val_accuracy": 1.0, "val_auroc": 1.0, "val_ap": 1

In [12]:

import pandas as pd
from IPython.display import display

rows = []

for variant in ["T0", "T1", "T2"]:
    for category in ["bottle", "hazelnut", "leather"]:
        files = list(
            (RUNS / variant / category / "seed42")
            .glob("full_*/summary.json")
        )

        completed = [
            f for f in files
            if json.loads(f.read_text()).get("status") == "completed"
            and (f.parent / "best.pt").is_file()
        ]

        assert len(completed) == 1, f"{variant}/{category} incomplet"

        summary = json.loads(completed[0].read_text())

        rows.append({
            "Variante": variant,
            "Catégorie": category,
            "Époques": summary["epochs_completed"],
            "Best epoch": summary["best_epoch"],
            "Val BCE": summary["best_val_loss"]
        })

df = pd.DataFrame(rows)
display(df)

assert len(df) == 9
backup_models()

print("9 modèles complets vérifiés.")
print("Archive finale :", ARCHIVE)


,Variante,Catégorie,Époques,Best epoch,Val BCE
0,T0,bottle,10,5,0.020394
1,T0,hazelnut,9,4,0.013522
2,T0,leather,13,8,0.000338
3,T1,bottle,20,20,0.069102
4,T1,hazelnut,20,20,0.112122
5,T1,leather,20,20,0.008702
6,T2,bottle,20,16,0.000069
7,T2,hazelnut,18,13,0.002080
8,T2,leather,14,9,0.000117


Archive actualisée : /kaggle/working/transfer_full_backup.zip
Taille Mo : 284.36
9 modèles complets vérifiés.
Archive finale : /kaggle/working/transfer_full_backup.zip
